# Entraînement du modèle de points clés du terrain

Produit `pitch_keypoints.pt`, le modèle qui manque au pipeline. Sans lui,
aucune statistique spatiale n'est calculable : ni radar, ni distance, ni possession.

**Durée** : 1 h 30 à 2 h 30 sur GPU T4 gratuit.

**Avant de lancer** : menu *Exécution → Modifier le type d'exécution → GPU*.

Trois cellules sont des garde-fous, pas des formalités. La cellule de
vérification arrête tout si l'ordre des points clés du dataset ne correspond
pas à celui du projet — sans elle, l'entraînement converge normalement et
produit un modèle qui prédit n'importe quoi.


## 1. Vérifier le GPU


In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout or
      "AUCUN GPU : Exécution -> Modifier le type d'exécution -> GPU")


## 2. Installer


In [ ]:
%pip install -q ultralytics roboflow pyyaml


## 3. Clé Roboflow

Saisie masquée : elle n'apparaît pas dans le notebook enregistré.


In [ ]:
from getpass import getpass
import os
os.environ['ROBOFLOW_API_KEY'] = getpass('Clé Roboflow : ')


## 4. Télécharger le dataset

`football-field-detection-f07vi` v15, CC BY 4.0 : 255 images d'entraînement,
34 de validation, 28 de test, annotées en 32 points clés.


In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
version = rf.workspace('roboflow-jvuqo').project('football-field-detection-f07vi').version(15)
dataset = version.download('yolov8', location='/content/pitch_ds', overwrite=False)
print('téléchargé dans', dataset.location)


## 5. Garde-fou : l'ordre des points clés

`flip_idx` indique à YOLO comment permuter les labels quand l'augmentation
`fliplr` mirore l'image. S'il ne correspond pas à la convention du projet,
le modèle apprendra des labels faux — en convergeant normalement. La panne
serait silencieuse et ne se verrait qu'à l'usage.

Cette liste a été vérifiée : elle est identique à celle que `geometry.py`
calcule par symétrie, sur les 32 entrées.


In [ ]:
import yaml
from pathlib import Path

ATTENDU = [24, 25, 26, 27, 28, 29, 22, 23, 21, 17, 18, 19, 20, 13, 14, 15,
           16, 9, 10, 11, 12, 8, 6, 7, 0, 1, 2, 3, 4, 5, 31, 30]

yaml_path = Path(dataset.location) / 'data.yaml'
cfg = yaml.safe_load(yaml_path.read_text())

assert cfg['kpt_shape'] == [32, 3], f"kpt_shape inattendu : {cfg['kpt_shape']}"
assert cfg['flip_idx'] == ATTENDU, (
    'ORDRE DES POINTS DIVERGENT : ne pas entraîner.\n'
    f"dataset : {cfg['flip_idx']}\nattendu : {ATTENDU}")
print('ordre des points clés conforme — entraînement autorisé')


## 6. Corriger les chemins

Roboflow écrit `train: ../train/images`. Ultralytics résout ce chemin
relativement au dossier du fichier, ce qui pointe un cran trop haut et fait
échouer l'entraînement au démarrage. On réécrit avec une racine explicite.


In [ ]:
cfg['path'] = str(Path(dataset.location).resolve())
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
cfg['test'] = 'test/images'
yaml_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

for split in ('train', 'valid', 'test'):
    n = len(list((Path(cfg['path']) / split / 'images').glob('*')))
    print(f'{split:6s}: {n} images')


## 7. Entraîner

Deux réglages sont critiques et n'ont pas de valeur par défaut correcte ici :

- **`mosaic=0`** — l'augmentation mosaïque colle quatre images en une. Le
  modèle apprendrait à chercher plusieurs terrains sur la même image, soit
  exactement le contraire du besoin.
- **`degrees`, `shear`, `perspective` à 0** — un terrain est un plan rigide.
  Ces déformations produiraient des géométries qui n'existent pas.

`fliplr=0.5` reste actif : c'est une vraie symétrie du terrain, et `flip_idx`
vérifié plus haut permute les labels correctement.

**Modèle** : `yolov8m-pose` par défaut. `yolov8x-pose` est plus précis mais
dépasse souvent la limite de session Colab gratuite. Commence par `m` ; si le
résultat de la cellule 9 ne suffit pas, recommence avec `x`.

`patience=50` arrête l'entraînement si rien ne progresse pendant 50 époques :
avec 255 images, le surapprentissage arrive bien avant la 300e.


In [ ]:
from ultralytics import YOLO

MODELE = 'yolov8m-pose.pt'   # passer à 'yolov8x-pose.pt' si besoin de précision

resultats = YOLO(MODELE).train(
    data=str(yaml_path),
    epochs=300,
    patience=50,
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='pitch',
    mosaic=0.0,       # non négociable, voir ci-dessus
    fliplr=0.5,
    degrees=0.0,
    shear=0.0,
    perspective=0.0,
    scale=0.3,
    plots=True,
)


## 8. Métriques d'entraînement


In [ ]:
best = Path('/content/runs/pitch/weights/best.pt')
print('poids :', best, '|', round(best.stat().st_size / 1e6, 1), 'Mo')

metriques = YOLO(str(best)).val(data=str(yaml_path), split='val')
print(f'mAP50 points clés   : {metriques.pose.map50:.3f}')
print(f'mAP50-95 points clés : {metriques.pose.map:.3f}')


## 9. Test d'acceptation : erreur en centimètres

Le mAP ne dit pas si le modèle est utilisable pour un produit. Ce qui compte,
c'est l'erreur de position **sur le terrain**, en centimètres.

On construit l'homographie à partir des points prédits, on projette les points
de vérité terrain, et on mesure l'écart. Repère : les annotations humaines
elles-mêmes tournent autour de 40 cm d'erreur résiduelle.

La géométrie ci-dessous doit rester identique à `pitch/geometry.py`. Ces
dimensions ne sont pas conventionnelles : elles ont été ajustées sur les 228
images annotées, l'optimum tombant sur le terrain FIFA standard.


In [ ]:
import cv2, numpy as np

L, W = 10500, 6800            # 105 x 68 m
PB_L, PB_W = 1650, 4032       # surface de réparation, loi du jeu
GB_L, GB_W = 550, 1832        # surface de but
R, SPOT = 915, 1100           # rond central, point de penalty

VERTICES = [
    (0,0), (0,(W-PB_W)//2), (0,(W-GB_W)//2), (0,(W+GB_W)//2), (0,(W+PB_W)//2), (0,W),
    (GB_L,(W-GB_W)//2), (GB_L,(W+GB_W)//2), (SPOT,W//2),
    (PB_L,(W-PB_W)//2), (PB_L,(W-GB_W)//2), (PB_L,(W+GB_W)//2), (PB_L,(W+PB_W)//2),
    (L//2,0), (L//2,W//2-R), (L//2,W//2+R), (L//2,W),
    (L-PB_L,(W-PB_W)//2), (L-PB_L,(W-GB_W)//2), (L-PB_L,(W+GB_W)//2), (L-PB_L,(W+PB_W)//2),
    (L-SPOT,W//2), (L-GB_L,(W-GB_W)//2), (L-GB_L,(W+GB_W)//2),
    (L,0), (L,(W-PB_W)//2), (L,(W-GB_W)//2), (L,(W+GB_W)//2), (L,(W+PB_W)//2), (L,W),
    (L//2-R,W//2), (L//2+R,W//2),
]
CIBLE = np.array(VERTICES, dtype=np.float32)
assert len(VERTICES) == len(set(VERTICES)) == 32

modele = YOLO(str(best))
SEUIL = 0.5      # même seuil que PitchConfig.confidence
MINI = 6         # même minimum que PitchConfig.min_keypoints

erreurs, exploitables, total = [], 0, 0
for label in sorted((Path(cfg['path'])/'valid'/'labels').glob('*.txt')):
    image = next(iter((Path(cfg['path'])/'valid'/'images').glob(label.stem + '.*')), None)
    if image is None:
        continue
    total += 1
    img = cv2.imread(str(image)); h, w = img.shape[:2]

    champs = label.read_text().split()
    if len(champs) < 5 + 96:
        continue
    verite = np.array(champs[5:5+96], float).reshape(32, 3)
    visible = verite[:, 2] > 0

    kp = modele.predict(img, verbose=False)[0].keypoints
    if kp is None or kp.xy is None or len(kp.xy) == 0:
        continue
    predit = kp.xy[0].cpu().numpy()
    confiance = kp.conf[0].cpu().numpy() if kp.conf is not None else np.ones(32)

    utilisable = (confiance >= SEUIL) & visible
    if utilisable.sum() < MINI:
        continue
    M, _ = cv2.findHomography(predit[utilisable], CIBLE[utilisable], cv2.RANSAC, 5.0)
    if M is None:
        continue
    exploitables += 1

    # Points de vérité terrain projetés avec l'homographie prédite.
    reel = np.stack([verite[:,0]*w, verite[:,1]*h], 1).astype(np.float32)[visible]
    proj = cv2.perspectiveTransform(reel.reshape(-1,1,2), M).reshape(-1,2)
    erreurs.append(np.median(np.linalg.norm(proj - CIBLE[visible], axis=1)))

e = np.array(erreurs)
print(f'images avec homographie exploitable : {exploitables}/{total} ({100*exploitables/total:.0f}%)')
print(f'erreur médiane   : {np.median(e):7.1f} cm')
print(f'90e centile      : {np.percentile(e,90):7.1f} cm')
print()
if np.median(e) < 100:
    print('BON — sous 1 m, exploitable pour des statistiques par équipe.')
elif np.median(e) < 300:
    print('PASSABLE — utilisable pour le radar, trop grossier pour les distances individuelles.')
else:
    print('INSUFFISANT — reprendre avec yolov8x-pose, ou plus d\'époques.')


## 10. Test sur tes propres images

Le dataset public vient surtout de diffusions européennes. La vraie question
est le taux d'images exploitables sur **tes** vidéos.

Repères mesurés visuellement, à confirmer ou infirmer ici :
Djoliba (caméra en tribune haute) ≥ 92 %, ASKO x Barracuda (bord de touche) 47 %.

Dépose des images extraites de tes matchs, puis exécute.


In [ ]:
from google.colab import files
import shutil

dossier = Path('/content/mes_images'); dossier.mkdir(exist_ok=True)
for nom in files.upload():
    shutil.move(nom, dossier / nom)

fichiers = sorted(p for p in dossier.iterdir() if p.suffix.lower() in ('.png','.jpg','.jpeg'))
compte = 0
for f in fichiers:
    kp = modele.predict(cv2.imread(str(f)), verbose=False)[0].keypoints
    n = 0
    if kp is not None and kp.conf is not None and len(kp.conf):
        n = int((kp.conf[0].cpu().numpy() >= SEUIL).sum())
    ok = n >= MINI
    compte += ok
    print(f'{f.name:40s} {n:2d} points >= {SEUIL}  {"exploitable" if ok else "-"}')

if fichiers:
    print(f'\nTaux exploitable : {compte}/{len(fichiers)} = {100*compte/len(fichiers):.0f}%')


## 11. Récupérer les poids

À déposer dans `models/pitch_keypoints.pt` du dépôt.


In [ ]:
files.download(str(best))
